#### reconstruct_inversions.ipynb
This notebook provides time-dependent visualizations of the TPW inversion results projected on paleogeographic reconstructions together with the flooding data themselves.

In [ ]:
import os
import numpy as np
import pandas as pd
import pygplates as pygp
import cartopy.crs as crs        
import matplotlib.pyplot as plt
import methods as m
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
### user-specified inputs ###
static_polygons = pygp.FeatureCollection(f'plate_model/static_polygons.gpml')      # static continental polygons
passive_margins = pygp.FeatureCollection(f'plate_model/passive_margins.gpml')      # assive margins polygons
foreland_basins = pygp.FeatureCollection(f'plate_model/foreland_basins.gpml')      # foreland basins polygons
rotation_model = f'plate_model/PHAB25_zero-lon-Africa.rot'                         # rotation file
anchor_plate = 998                                                                 # anchor plate ID: 998 = paleomagnetic frame with zero-longitude Africa assumption

start = 0    # start time (in Ma) = youngest reconstruction step
stop = 320     # stop time (in Ma) = oldest reconstruction step
step = 10      # temporal step size (in Ma)

show_filtered = False
apply_glacial_filter = False # if True: observations from Antarctica since 30 Ma are removed

invdir = f'data/inversions'        # location of permutation results directory
figdir = f'figs/reconstructions'   # directory to output reconstruction figs
os.makedirs(figdir, exist_ok=True)

In [ ]:
# make reference grid for plotting Y21 reference pattern
reflons = np.linspace(-180, 180, 181)
reflats = np.linspace(-90, 90, 91)
rlons, rlats = np.meshgrid(reflons, reflats)

for t in range(start, stop, step):
    print (f'processing interval {t}-{t+step} Ma')
    t0 = pd.read_csv(f'obs_grids/{t}_Ma.csv')                # read in observation grids
    t1 = pd.read_csv(f'obs_grids/{t+step}_Ma.csv')
    merged = pd.merge(t0, t1, on='name', suffixes=('_t0', '_t1'))
    merged = merged.dropna(subset=['state_t0', 'state_t1'])                      # drop NaNs
    merged['diff'] =  merged['state_t1'] - merged['state_t0']                    # find points where there is flux: -1 = regression / 1 = transgression

    inv = pd.read_csv(f'{invdir}/{t}_Ma.csv')         # import inversion results 
    mask = inv['beta'] > 0                            # consider only results where TPW coefficient is positive                          
    best_idx = inv.loc[mask, 'llr'].idxmax()          # find best-fit
    lam = inv.loc[best_idx, 'lambda']                 # best-fitting TPW axis location
    s = m.Y21(lam, rlats, rlons)                      # generate reference Y21 pattern for the TPW axis location

    recon_t0 = m.reconstruct_polygons(static_polygons, rotation_model, t, anchor_plate=anchor_plate)    # reconstruct polygons @ t0
    recon_t1 = m.reconstruct_polygons(static_polygons, rotation_model, t+step, anchor_plate=anchor_plate)    # reconstruct polygons @ t0

    if show_filtered:
        fig = plt.figure(figsize=(14, 15))
        gs = fig.add_gridspec(4, 6, height_ratios=[1, 2, 0.2, 1.5], hspace=0, wspace=0)
    else:
        fig = plt.figure(figsize=(14, 10))
        gs = fig.add_gridspec(2, 6, height_ratios=[1, 2], hspace=0, wspace=0)
    ax1 = fig.add_subplot(gs[0, 0:3], projection=crs.Robinson())
    m.plot_observations(t+step, recon_t1, t1, ax=ax1) 
    ax1.set_title(f'Flooding at {t+step} Ma', pad=5, fontsize=12)

    ax2 = fig.add_subplot(gs[0, 3:6], projection=crs.Robinson())
    m.plot_observations(t, recon_t0, t0, ax=ax2) 
    ax2.set_title(f'Flooding at {t} Ma', pad=5, fontsize=12)

    ax3 = fig.add_subplot(gs[1, 1:5], projection=crs.Robinson())
    m.plot_data_and_model(recon_t0, lam, rlats, rlons, s, merged, ax=ax3)
    ax3.set_title(f'Sea-level change {t+step}–{t} Ma', pad=5, fontsize=13)
    ax3.legend(loc='lower right', bbox_to_anchor=(1.15, -0.1), frameon=False)

    handles, labels = ax2.get_legend_handles_labels()
    if show_filtered:
        filtered = merged[~(merged['in_margin_t0'] | merged['in_margin_t1'] | merged['in_basin_t0']  | merged['in_basin_t1'])]  # get rid of any 'True' values
        if apply_glacial_filter and t <= 30:
            filtered = filtered[filtered['lat_t0'] >= -60] # remove all points above -60 S latitude

        f_inv = pd.read_csv(f'{invdir}/filtered/{t}_Ma.csv')           # import filtered inversion results 
        f_mask = f_inv['beta'] > 0                              # consider only results where TPW coefficient is positive                          
        f_best_idx = f_inv.loc[f_mask, 'llr'].idxmax()          # find best-fit
        f_lam = f_inv.loc[f_best_idx, 'lambda']                 # best-fitting TPW axis location
        f_s = m.Y21(f_lam, rlats, rlons)                        # generate reference Y21 pattern for the TPW axis location

        margins_t0 = m.reconstruct_polygons(passive_margins, rotation_model, t, anchor_plate=anchor_plate)    # reconstruct polygons @ t0
        basins_t0 = m.reconstruct_polygons(foreland_basins, rotation_model, t, anchor_plate=anchor_plate)    # reconstruct polygons @ t0

        ax4 = fig.add_subplot(gs[3, 1:5], projection=crs.Robinson())
        _, handles_ax4 = m.plot_data_and_model_wfeatures(recon_t0,  margins_t0, basins_t0, f_lam, rlats, rlons, f_s, filtered, ax=ax4)       
        ax4.set_title(f'Filtered result {t+step}–{t} Ma', pad=5, fontsize=13)
        ax4.legend(handles=handles_ax4, loc='lower right', bbox_to_anchor=(1.15, 0.0), frameon=False)

        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.50, 0.78), ncol=1, markerscale=2, frameon=False)
    else: 
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.50, 0.69), ncol=1, markerscale=2, frameon=False)
    
    fig.subplots_adjust(top=0.95, bottom=0.05, left=0.05, right=0.95)
    plt.savefig(f'{figdir}/{t}_Ma.png')
    plt.savefig(f'{figdir}/{t}_Ma.pdf')
    #plt.show()
    plt.close()